# 02 | Set the weights once and hold

Use only 2006 daily returns to decide weights. Invest $100,000 at the first fully marked close of 2007. Subsequent valuations use the same shares in entry-date units, so portfolio weights drift.

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd
from scipy.optimize import minimize

private = Path("data/private")
results = Path("results")
prices = pd.read_csv(private / "usd_prices.csv", index_col="date", parse_dates=True)
daily_returns = prices.pct_change(fill_method=None)
training = daily_returns.loc["2006"].dropna()
covariance = training.cov()
assert len(training) > 200 and covariance.shape == (10, 10)
display(covariance.round(6))

,GS,UBS,JPM,C,BCS,MS,DB,BAC,BNP,CS
GS,0.000218,0.000112,0.000094,0.000065,0.000085,0.000145,0.000111,0.000051,0.000058,0.000112
UBS,0.000112,0.000193,0.000085,0.000056,0.000108,0.000107,0.000148,0.000057,0.000112,0.000168
JPM,0.000094,0.000085,0.000115,0.000064,0.000064,0.000085,0.000089,0.000060,0.000042,0.000081
C,0.000065,0.000056,0.000064,0.000082,0.000051,0.000059,0.000060,0.000046,0.000031,0.000051
BCS,0.000085,0.000108,0.000064,0.000051,0.000153,0.000080,0.000114,0.000040,0.000084,0.000112
MS,0.000145,0.000107,0.000085,0.000059,0.000080,0.000158,0.000108,0.000053,0.000065,0.000107
DB,0.000111,0.000148,0.000089,0.000060,0.000114,0.000108,0.000191,0.000059,0.000119,0.000162
BAC,0.000051,0.000057,0.000060,0.000046,0.000040,0.000053,0.000059,0.000065,0.000033,0.000052
BNP,0.000058,0.000112,0.000042,0.000031,0.000084,0.000065,0.000119,0.000033,0.000214,0.000109
CS,0.000112,0.000168,0.000081,0.000051,0.000112,0.000107,0.000162,0.000052,0.000109,0.000242


## Minimum-variance allocation

Optimize sample daily variance with full investment and a 1% floor. Equal weights are a feasible starting point. Scale the objective by 10,000 because daily decimal-return variance is small. Use the analytic gradient and verify the solver output against equal weights.

In [2]:
cov = covariance.to_numpy()
equal_weights = np.full(prices.shape[1], 1 / prices.shape[1])

def scaled_variance(weights):
    return 10_000 * (weights @ cov @ weights)

optimum = minimize(
    scaled_variance,
    equal_weights,
    jac=lambda weights: 20_000 * (cov @ weights),
    method="SLSQP",
    bounds=[(0.01, 1.0)] * len(equal_weights),
    constraints={"type": "eq", "fun": lambda weights: weights.sum() - 1,
                 "jac": lambda weights: np.ones_like(weights)},
    options={"ftol": 1e-12, "maxiter": 1_000},
)
assert optimum.success, optimum.message
assert np.isclose(optimum.x.sum(), 1, rtol=0, atol=1e-10)
assert optimum.x.min() >= 0.01 - 1e-10
assert scaled_variance(optimum.x) <= scaled_variance(equal_weights) + 1e-10

weights = pd.Series(optimum.x, index=prices.columns, name="entry_weight")
display(weights.map("{:.2%}".format).rename_axis("bank_code").to_frame())
print(f"2006 complete daily returns: {len(training)}")
print(f"2006 sample volatility: {np.sqrt(optimum.fun / 10000):.3%} optimized, "
      f"{np.sqrt(scaled_variance(equal_weights) / 10000):.3%} equal weight")

,entry_weight
bank_code,
GS,1.00%
UBS,1.00%
JPM,1.00%
C,27.13%
BCS,4.35%
MS,1.00%
DB,1.00%
BAC,53.01%
BNP,9.51%


2006 complete daily returns: 245
2006 sample volatility: 0.747% optimized, 0.956% equal weight


## Make the trade and reconcile daily P&L

The split-adjusted series is rescaled to the actual entry quote where a share factor changed before entry. Fixed shares are in entry-date units. A missing mark leaves that day's portfolio value unavailable. Differencing the full calendar before filtering keeps a gap from becoming a one-session return.

In [3]:
entry_date = prices.loc["2007"].dropna().index[0]
holding_prices = prices.loc[entry_date:].copy()
us_raw = pd.read_csv(private / "us_closes.csv", parse_dates=["date"])
entry_us = us_raw.loc[us_raw["date"].eq(entry_date)].set_index("permno")["prc"]
coverage = pd.read_csv(results / "data_coverage.csv", index_col="bank_code")
for code, security_id in coverage.loc[coverage["venue"].eq("NYSE"), "security_id"].items():
    permno = int(security_id.split()[-1])
    holding_prices[code] *= entry_us.loc[permno] / holding_prices.loc[entry_date, code]

capital = 100_000.0
shares = capital * weights / holding_prices.loc[entry_date]
position_values = holding_prices * shares
bank_pnl = position_values.diff()
nav = position_values.sum(axis=1, min_count=len(weights)).rename("portfolio_value_usd")
portfolio_pnl = nav.diff().rename("portfolio_pnl_usd")
portfolio_returns = nav.pct_change(fill_method=None).rename("portfolio_return")
complete = portfolio_pnl.notna()

assert np.isclose(nav.loc[entry_date], capital, rtol=0, atol=1e-8)
assert complete.equals(bank_pnl.notna().all(axis=1))
np.testing.assert_allclose(portfolio_pnl[complete], bank_pnl.loc[complete].sum(axis=1), atol=1e-8)
np.testing.assert_allclose(
    portfolio_pnl[complete], (nav.shift() * portfolio_returns)[complete], atol=1e-8
)
implied_shares = position_values.div(holding_prices).stack()
np.testing.assert_allclose(implied_shares, shares.reindex(implied_shares.index.get_level_values(1)),
                           rtol=1e-12)
assert not position_values.div(nav, axis=0).iloc[-1].round(8).equals(weights.round(8))

positions = pd.DataFrame({"entry_weight": weights, "entry_dollars": capital * weights,
                          "end_weight": position_values.iloc[-1] / nav.iloc[-1]})
positions.rename_axis("bank_code").to_csv(results / "allocation.csv")
display(positions.round(4))

,entry_weight,entry_dollars,end_weight
GS,0.0100,1000.0000,0.0177
UBS,0.0100,1000.0000,0.0103
JPM,0.0100,1000.0000,0.0276
C,0.2713,27127.3631,0.1386
BCS,0.0435,4348.4061,0.0302
MS,0.0100,1000.0000,0.0083
DB,0.0100,1000.0000,0.0126
BAC,0.5301,53012.6926,0.5888
BNP,0.0951,9511.5382,0.1490
CS,0.0100,1000.0000,0.0170


## Save the portfolio for the backtest

The daily bank-level ledger stays in the local cache. P&L and bank returns are defined on consecutive trading sessions only. Market stress was already present in 2007, which serves as the initialization period.

In [4]:
portfolio = pd.concat([nav, portfolio_pnl, portfolio_returns], axis=1)
portfolio.to_csv(private / "portfolio.csv")
position_values.to_csv(private / "bank_values.csv")
bank_pnl.to_csv(private / "bank_pnl.csv")
holding_prices.pct_change(fill_method=None).to_csv(private / "bank_returns.csv")

periods = pd.DataFrame(
    {"entry_date": [entry_date], "entry_value_usd": [nav.iloc[0]], "end_value_usd": [nav.iloc[-1]],
     "optimization_days": [len(training)], "initialization_returns": [portfolio_returns.loc["2007"].count()],
     "optimized_daily_vol": [np.sqrt(optimum.fun / 10000)],
     "equal_weight_daily_vol": [np.sqrt(scaled_variance(equal_weights) / 10000)]}
)
periods.to_csv(results / "portfolio_summary.csv", index=False)
display(periods)

,entry_date,entry_value_usd,end_value_usd,optimization_days,initialization_returns,optimized_daily_vol,equal_weight_daily_vol
0,2007-01-03,100000.0,23770.773373,245,244,0.007469,0.009557
